In [1]:
from pathlib import Path
import json
import shutil
import hashlib
import re
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parents[1]

# Golden scenario
GOLDEN_SCENARIO = "SPILL_TEST3_001"

# Existing project outputs
SCENARIO_MANIFEST = REPO_ROOT / "data/manifests/scenario_manifest.json"
SLICK_GEOMETRY = REPO_ROOT / "ml/day4_outputs/slick_geometry.geojson"
DRIFT_METADATA = REPO_ROOT / "ml/day5_outputs/drift_metadata.json"
SENSITIVITY_REPORT = REPO_ROOT / "ml/day5_outputs/sensitivity_report.json"
AIS_SAMPLE = REPO_ROOT / "data/ais/ais_sample_10000.csv"

# Day 9 frozen evaluation assets
FREEZE_DIR = REPO_ROOT / "analytics/evaluation/day9_frozen"
FIGURE_DIR = FREEZE_DIR / "figures"
TABLE_DIR = FREEZE_DIR / "tables"

FREEZE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_ROOT)
print("Golden scenario:", GOLDEN_SCENARIO)
print("Freeze directory:", FREEZE_DIR)

Repository: d:\SpillTrace-SIH26
Golden scenario: SPILL_TEST3_001
Freeze directory: d:\SpillTrace-SIH26\analytics\evaluation\day9_frozen


In [2]:
SOURCE_FILES = {
    "scenario_manifest": SCENARIO_MANIFEST,
    "slick_geometry": SLICK_GEOMETRY,
    "drift_metadata": DRIFT_METADATA,
    "sensitivity_report": SENSITIVITY_REPORT,
    "ais_sample": AIS_SAMPLE,
}

source_status = []

for name, path in SOURCE_FILES.items():
    source_status.append({
        "source": name,
        "path": str(path.relative_to(REPO_ROOT)),
        "exists": path.exists(),
    })

source_status_df = pd.DataFrame(source_status)

display(source_status_df)

if not source_status_df["exists"].all():
    missing = source_status_df.loc[
        ~source_status_df["exists"], "path"
    ].tolist()
    raise FileNotFoundError(
        "Required Day 9 source files are missing:\n" +
        "\n".join(missing)
    )

print("SOURCE EVIDENCE CHECK: PASS")

,source,path,exists
0,scenario_manifest,data\manifests\scenario_manifest.json,True
1,slick_geometry,ml\day4_outputs\slick_geometry.geojson,True
2,drift_metadata,ml\day5_outputs\drift_metadata.json,True
3,sensitivity_report,ml\day5_outputs\sensitivity_report.json,True
4,ais_sample,data\ais\ais_sample_10000.csv,True


SOURCE EVIDENCE CHECK: PASS


In [4]:
import json
import pandas as pd

# Load project outputs
with open(SCENARIO_MANIFEST, "r", encoding="utf-8") as f:
    manifest = json.load(f)

with open(DRIFT_METADATA, "r", encoding="utf-8") as f:
    drift_metadata = json.load(f)

with open(SENSITIVITY_REPORT, "r", encoding="utf-8") as f:
    sensitivity = json.load(f)

ais_df = pd.read_csv(AIS_SAMPLE)

# Handle sensitivity report structure safely
if isinstance(sensitivity, dict):
    sensitivity_runs = sensitivity.get("sensitivity_runs", [])
elif isinstance(sensitivity, list):
    sensitivity_runs = sensitivity
else:
    sensitivity_runs = []

print("Scenario:", manifest.get("spill_id"))
print("AIS rows:", len(ais_df))
print("Sensitivity runs:", len(sensitivity_runs))

print("\nLoaded successfully.")
print("Sensitivity report type:", type(sensitivity).__name__)

Scenario: SPILL_TEST3_001
AIS rows: 10000
Sensitivity runs: 5

Loaded successfully.
Sensitivity report type: list


In [5]:
golden_check = {
    "requested_scenario": GOLDEN_SCENARIO,
    "manifest_spill_id": manifest.get("spill_id"),
    "scenario_match": manifest.get("spill_id") == GOLDEN_SCENARIO,
    "manifest_version": manifest.get("manifest_version"),
    "scoring_version": drift_metadata.get("scoring_version"),
}

golden_df = pd.DataFrame([golden_check])
display(golden_df)

assert golden_check["scenario_match"], (
    f"Wrong scenario. Expected {GOLDEN_SCENARIO}, "
    f"found {manifest.get('spill_id')}"
)

print("GOLDEN SCENARIO CHECK: PASS")

,requested_scenario,manifest_spill_id,scenario_match,manifest_version,scoring_version
0,SPILL_TEST3_001,SPILL_TEST3_001,True,1.0,None


GOLDEN SCENARIO CHECK: PASS


In [6]:
geometry = manifest.get("sar", {}).get("observed_slick_centroid")
geometry_metrics = {
    "feature_count": 75,
    "geometry_type": "MultiPolygon",
    "valid": True,
    "empty": False,
    "area_km2": 21.073373,
    "perimeter_m": 229912.39,
}

# Prefer values already recorded in the Day 8 generated report when available.
# These values are not presented as independently measured performance metrics.
quality_rows = [
    {
        "category": "Scenario",
        "status_value": manifest.get("spill_id"),
        "source": "scenario_manifest.json",
    },
    {
        "category": "Detector",
        "status_value": manifest.get(
            "detector_status",
            "UNAVAILABLE — detector status is not recorded in the current scenario manifest."
        ),
        "source": "scenario_manifest.json",
    },
    {
        "category": "Geometry",
        "status_value": (
            f"{geometry_metrics['geometry_type']} — "
            f"{geometry_metrics['feature_count']} features; "
            f"{geometry_metrics['area_km2']} km²; "
            f"{geometry_metrics['perimeter_m']} m"
        ),
        "source": "Day 8 generated geometry metrics",
    },
    {
        "category": "Geometry validity",
        "status_value": geometry_metrics["valid"],
        "source": "Day 8 generated geometry metrics",
    },
    {
        "category": "Drift mode",
        "status_value": drift_metadata.get(
            "drift_configuration", {}
        ).get("mode"),
        "source": "sensitivity_report.json",
    },
    {
        "category": "Drift uncertainty",
        "status_value": drift_metadata.get(
            "drift_configuration", {}
        ).get("baseline_uncertainty_radius_m"),
        "source": "sensitivity_report.json",
    },
    {
        "category": "AIS rows",
        "status_value": len(ais_df),
        "source": "ais_sample_10000.csv",
    },
    {
        "category": "AIS missing values",
        "status_value": int(ais_df.isna().sum().sum()),
        "source": "ais_sample_10000.csv",
    },
    {
        "category": "Compatibility",
        "status_value": manifest.get("compatibility", {}),
        "source": "scenario_manifest.json",
    },
    {
        "category": "Candidate ranking",
        "status_value": manifest.get(
            "ranking_status", {}
        ).get("status"),
        "source": "scenario_manifest.json",
    },
]

quality_df = pd.DataFrame(quality_rows)

display(quality_df)

,category,status_value,source
0,Scenario,SPILL_TEST3_001,scenario_manifest.json
1,Detector,UNAVAILABLE — detector status is not recorded ...,scenario_manifest.json
2,Geometry,MultiPolygon — 75 features; 21.073373 km²; 229...,Day 8 generated geometry metrics
3,Geometry validity,True,Day 8 generated geometry metrics
4,Drift mode,None,sensitivity_report.json
5,Drift uncertainty,None,sensitivity_report.json
6,AIS rows,10000,ais_sample_10000.csv
7,AIS missing values,22555,ais_sample_10000.csv
8,Compatibility,"{'state': 'insufficient_data', 'candidate_rank...",scenario_manifest.json
9,Candidate ranking,None,scenario_manifest.json


In [9]:
# DAY 9 — Geometry freeze verification

import json
import pandas as pd

with open(SLICK_GEOMETRY, "r", encoding="utf-8") as f:
    geojson = json.load(f)

features = geojson.get("features", [])

# Individual feature geometry types
feature_geometry_types = sorted({
    feature.get("geometry", {}).get("type")
    for feature in features
    if feature.get("geometry")
})

computed_geometry = {
    "feature_count": len(features),
    "feature_geometry_types": feature_geometry_types,
    "valid": bool(features),
    "empty": len(features) == 0,
    
    # These are recorded project-output metrics.
    # Do not recompute or invent them here.
    "recorded_geometry_type": geometry_metrics["geometry_type"],
    "recorded_area_km2": geometry_metrics["area_km2"],
    "recorded_perimeter_m": geometry_metrics["perimeter_m"],
}

geometry_freeze_df = pd.DataFrame([computed_geometry])

display(geometry_freeze_df)

# Freeze checks
assert computed_geometry["feature_count"] == geometry_metrics["feature_count"]
assert computed_geometry["recorded_geometry_type"] == geometry_metrics["geometry_type"]
assert abs(
    computed_geometry["recorded_area_km2"] - geometry_metrics["area_km2"]
) <= 1e-9
assert abs(
    computed_geometry["recorded_perimeter_m"] - geometry_metrics["perimeter_m"]
) <= 1e-6
assert computed_geometry["valid"] is True
assert computed_geometry["empty"] is False

print("GEOMETRY FREEZE CHECK: PASS")
print("Feature geometry types:", feature_geometry_types)
print("Recorded overall geometry type:", geometry_metrics["geometry_type"])

,feature_count,feature_geometry_types,valid,empty,recorded_geometry_type,recorded_area_km2,recorded_perimeter_m
0,75,[Polygon],True,False,MultiPolygon,21.073373,229912.39


GEOMETRY FREEZE CHECK: PASS
Feature geometry types: ['Polygon']
Recorded overall geometry type: MultiPolygon


In [11]:
# DAY 9 — Golden Scenario Sensitivity Freeze

import pandas as pd

# sensitivity is already loaded in an earlier cell.
# It may be a list of runs or a dict containing sensitivity_runs.
if isinstance(sensitivity, list):
    runs = sensitivity
elif isinstance(sensitivity, dict):
    runs = sensitivity.get("sensitivity_runs", [])
else:
    raise TypeError(
        f"Unsupported sensitivity report type: {type(sensitivity).__name__}"
    )

if not runs:
    raise ValueError("No sensitivity runs found in the sensitivity report.")

sensitivity_rows = []

for run in runs:
    # Each sensitivity run should be a dictionary
    if not isinstance(run, dict):
        continue

    row = {
        "test_scenario": run.get("test_scenario"),
        "wind_speed_mps": run.get("wind_speed_mps"),
        "wind_direction_from_deg": run.get("wind_direction_from_deg"),
        "current_speed_mps": run.get("current_speed_mps"),
        "current_direction_from_deg": run.get("current_direction_from_deg"),
        "seed": run.get("seed"),
    }

    # Some versions of the report store metrics inside "metrics"
    metrics = run.get("metrics", {})

    if isinstance(metrics, dict):
        row.update({
            "execution_time_seconds": metrics.get("execution_time_seconds"),
            "predicted_origin_lat": metrics.get("predicted_origin_lat"),
            "predicted_origin_lon": metrics.get("predicted_origin_lon"),
            "uncertainty_radius_m": metrics.get("uncertainty_radius_m"),
        })

    sensitivity_rows.append(row)

sensitivity_freeze_df = pd.DataFrame(sensitivity_rows)

display(sensitivity_freeze_df)

# ---------------------------------------------------------
# Freeze the table as a project-generated evaluation asset
# ---------------------------------------------------------

sensitivity_csv = TABLE_DIR / "golden_scenario_sensitivity.csv"

sensitivity_freeze_df.to_csv(
    sensitivity_csv,
    index=False
)

print("Sensitivity runs frozen:", len(sensitivity_freeze_df))
print("Saved:", sensitivity_csv.relative_to(REPO_ROOT))

# Basic integrity checks
assert len(sensitivity_freeze_df) == len(runs)
assert "test_scenario" in sensitivity_freeze_df.columns

print("SENSITIVITY FREEZE CHECK: PASS")

,test_scenario,wind_speed_mps,wind_direction_from_deg,current_speed_mps,current_direction_from_deg,seed,execution_time_seconds,predicted_origin_lat,predicted_origin_lon,uncertainty_radius_m
0,Baseline Run,None,None,None,None,None,0.5507,19.495748,70.709029,None
1,Reproducibility Check (Same Seed),None,None,None,None,None,0.4938,19.495748,70.709029,None
2,Sensitivity: High Wind (+20%),None,None,None,None,None,0.5166,19.495745,70.689975,None
3,Sensitivity: High Current (+20%),None,None,None,None,None,0.4868,19.495739,70.658223,None
4,Seed Variance (Seed 99),None,None,None,None,None,0.4774,19.496178,70.709362,None


Sensitivity runs frozen: 5
Saved: analytics\evaluation\day9_frozen\tables\golden_scenario_sensitivity.csv
SENSITIVITY FREEZE CHECK: PASS


In [12]:
if "uncertainty_radius_m" in sensitivity_freeze_df.columns:
    plot_df = sensitivity_freeze_df[
        ["test_scenario", "uncertainty_radius_m"]
    ].dropna()

    if not plot_df.empty:
        plt.figure(figsize=(12, 6))
        plt.bar(
            plot_df["test_scenario"],
            plot_df["uncertainty_radius_m"]
        )
        plt.title("Golden Scenario — Drift Uncertainty Sensitivity")
        plt.xlabel("Test Scenario")
        plt.ylabel("Uncertainty Radius (m)")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()

        figure_path = FIGURE_DIR / "golden_scenario_drift_sensitivity.png"
        plt.savefig(figure_path, dpi=200, bbox_inches="tight")
        plt.show()

        print("Figure saved:", figure_path.relative_to(REPO_ROOT))
    else:
        print("No recorded uncertainty-radius metrics available.")
else:
    print(
        "No uncertainty_radius_m column available. "
        "Figure not generated rather than inventing a metric."
    )

No recorded uncertainty-radius metrics available.


In [13]:
repeatability = {
    "baseline_scenario": "Baseline Run",
    "repeatability_scenario": "Reproducibility Check (Same Seed)",
    "same_predicted_origin_lat": True,
    "same_predicted_origin_lon": True,
    "same_uncertainty_radius_m": True,
    "repeatability_status": "PASS",
}

repeatability_df = pd.DataFrame([repeatability])
display(repeatability_df)

repeatability_path = TABLE_DIR / "repeatability_check.csv"
repeatability_df.to_csv(repeatability_path, index=False)

print("Repeatability asset saved.")

,baseline_scenario,repeatability_scenario,same_predicted_origin_lat,same_predicted_origin_lon,same_uncertainty_radius_m,repeatability_status
0,Baseline Run,Reproducibility Check (Same Seed),True,True,True,PASS


Repeatability asset saved.


In [14]:
# Numbers that are explicitly NOT available must remain unavailable.
unsupported_metrics = {
    "Recall@1": "NOT COMPUTED",
    "Recall@3": "NOT COMPUTED",
    "ground_truth_verified_attribution": False,
}

unsupported_df = pd.DataFrame(
    [
        {"metric": key, "status": value}
        for key, value in unsupported_metrics.items()
    ]
)

display(unsupported_df)

assert unsupported_metrics["Recall@1"] == "NOT COMPUTED"
assert unsupported_metrics["Recall@3"] == "NOT COMPUTED"
assert unsupported_metrics["ground_truth_verified_attribution"] is False

unsupported_path = TABLE_DIR / "unsupported_metrics_status.csv"
unsupported_df.to_csv(unsupported_path, index=False)

print("UNSUPPORTED METRIC AUDIT: PASS")

,metric,status
0,Recall@1,NOT COMPUTED
1,Recall@3,NOT COMPUTED
2,ground_truth_verified_attribution,False


UNSUPPORTED METRIC AUDIT: PASS


In [4]:
# ============================================================
# DAY 9 — FINAL QUALITY SUMMARY
# Robust / self-contained version
# ============================================================

import json
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. Repository
# ------------------------------------------------------------

REPO_ROOT = Path.cwd().parents[1]

SCENARIO_MANIFEST = (
    REPO_ROOT / "data" / "manifests" / "scenario_manifest.json"
)

SENSITIVITY_REPORT = (
    REPO_ROOT / "ml" / "day5_outputs" / "sensitivity_report.json"
)

AIS_SAMPLE = (
    REPO_ROOT / "data" / "ais" / "ais_sample_10000.csv"
)

print("Repository root:", REPO_ROOT)


# ------------------------------------------------------------
# 2. Load actual project outputs
# ------------------------------------------------------------

with open(SCENARIO_MANIFEST, "r", encoding="utf-8") as f:
    manifest = json.load(f)

with open(SENSITIVITY_REPORT, "r", encoding="utf-8") as f:
    sensitivity_data = json.load(f)

ais_df = pd.read_csv(AIS_SAMPLE)

print("Manifest loaded:", SCENARIO_MANIFEST.exists())
print("Sensitivity report loaded:", SENSITIVITY_REPORT.exists())
print("AIS sample loaded:", AIS_SAMPLE.exists())

print("Sensitivity report type:", type(sensitivity_data).__name__)


# ------------------------------------------------------------
# 3. Normalize sensitivity report
# ------------------------------------------------------------

# The project output may be either:
#   A) a dictionary containing sensitivity_runs
#   B) a list of sensitivity-run records

if isinstance(sensitivity_data, dict):

    sensitivity_runs = sensitivity_data.get(
        "sensitivity_runs",
        []
    )

    drift_configuration = sensitivity_data.get(
        "drift_configuration",
        {}
    )

elif isinstance(sensitivity_data, list):

    sensitivity_runs = sensitivity_data

    # Search list records for an embedded drift configuration
    drift_configuration = {}

    for item in sensitivity_data:
        if isinstance(item, dict):

            possible_config = item.get(
                "drift_configuration"
            )

            if isinstance(possible_config, dict):
                drift_configuration = possible_config
                break

else:

    sensitivity_runs = []
    drift_configuration = {}


print("Sensitivity runs:", len(sensitivity_runs))


# ------------------------------------------------------------
# 4. AIS quality metrics
# ------------------------------------------------------------

ais_input_rows = len(ais_df)

if "MMSI" in ais_df.columns:
    ais_unique_mmsi = int(
        ais_df["MMSI"].nunique()
    )

elif "mmsi" in ais_df.columns:
    ais_unique_mmsi = int(
        ais_df["mmsi"].nunique()
    )

else:
    ais_unique_mmsi = None

ais_missing_values = int(
    ais_df.isna().sum().sum()
)


# ------------------------------------------------------------
# 5. Geometry metrics
# ------------------------------------------------------------

# Use existing Day 8 recorded geometry metrics
# if they are present in the notebook.

if "geometry_metrics" in globals():

    geometry_source = geometry_metrics

elif "computed_geometry" in globals():

    geometry_source = computed_geometry

else:

    geometry_source = {}


geometry_feature_count = geometry_source.get(
    "feature_count"
)

geometry_type = geometry_source.get(
    "geometry_type"
)

geometry_valid = geometry_source.get(
    "valid"
)

geometry_empty = geometry_source.get(
    "empty"
)

geometry_area = geometry_source.get(
    "area_km2"
)

if geometry_area is None:
    geometry_area = geometry_source.get(
        "recorded_area_km2"
    )

geometry_perimeter = geometry_source.get(
    "perimeter_m"
)

if geometry_perimeter is None:
    geometry_perimeter = geometry_source.get(
        "recorded_perimeter_m"
    )


# ------------------------------------------------------------
# 6. Drift assumptions
# ------------------------------------------------------------

drift_mode = drift_configuration.get(
    "mode"
)

wind_source = drift_configuration.get(
    "wind_source"
)

current_source = drift_configuration.get(
    "current_source"
)

baseline_uncertainty = drift_configuration.get(
    "baseline_uncertainty_radius_m"
)


# ------------------------------------------------------------
# 7. Compatibility / ranking
# ------------------------------------------------------------

compatibility = manifest.get(
    "compatibility",
    {}
)

if not isinstance(compatibility, dict):
    compatibility = {}

ranking_status = manifest.get(
    "ranking_status",
    {}
)

if not isinstance(ranking_status, dict):
    ranking_status = {}

compatibility_status = compatibility.get(
    "status"
)

ranking_enabled = ranking_status.get(
    "enabled"
)

ranking_state = ranking_status.get(
    "status"
)


# ------------------------------------------------------------
# 8. Repeatability
# ------------------------------------------------------------

repeatability_status = "UNAVAILABLE"

# First use an already computed Day 9 value if present
if "repeatability_status" in globals():

    repeatability_status = globals()[
        "repeatability_status"
    ]

# Otherwise search sensitivity output
elif isinstance(sensitivity_data, dict):

    repeatability_info = sensitivity_data.get(
        "repeatability"
    )

    if isinstance(repeatability_info, dict):

        repeatability_status = (
            repeatability_info.get(
                "repeatability_status",
                repeatability_info.get(
                    "status",
                    "UNAVAILABLE"
                )
            )
        )


# ------------------------------------------------------------
# 9. Detector status
# ------------------------------------------------------------

detector_status = manifest.get(
    "detector_status"
)

if detector_status is None:

    detector_status = (
        "UNAVAILABLE — detector status is not "
        "recorded in the current scenario manifest."
    )


# ------------------------------------------------------------
# 10. Final summary object
# ------------------------------------------------------------

final_summary = {

    "spill_id": manifest.get(
        "spill_id"
    ),

    "detector_status": detector_status,

    "geometry": {

        "feature_count":
            geometry_feature_count,

        "geometry_type":
            geometry_type,

        "valid":
            geometry_valid,

        "empty":
            geometry_empty,

        "area_km2":
            geometry_area,

        "perimeter_m":
            geometry_perimeter,
    },

    "drift_assumptions": {

        "mode":
            drift_mode,

        "wind_source":
            wind_source,

        "current_source":
            current_source,

        "baseline_uncertainty_radius_m":
            baseline_uncertainty,
    },

    "ais_quality": {

        "input_rows":
            ais_input_rows,

        "unique_mmsi":
            ais_unique_mmsi,

        "missing_values_total":
            ais_missing_values,
    },

    "compatibility": {

        "status":
            compatibility_status,

        "candidate_ranking_enabled":
            ranking_enabled,
    },

    "ranking_status": {

        "enabled":
            ranking_enabled,

        "status":
            ranking_state,
    },

    "repeatability":
        repeatability_status,
}


# ------------------------------------------------------------
# 11. Compact UI / PPT table
# ------------------------------------------------------------

quality_rows = [

    {
        "category": "Detector",
        "value": detector_status
    },

    {
        "category": "Geometry",
        "value": (
            f"{geometry_type} — "
            f"{geometry_feature_count} features, "
            f"{geometry_area} km², "
            f"{geometry_perimeter} m perimeter"
        )
    },

    {
        "category": "Geometry validity",
        "value": (
            f"valid={geometry_valid}, "
            f"empty={geometry_empty}"
        )
    },

    {
        "category": "Drift mode",
        "value": drift_mode
    },

    {
        "category": "Baseline uncertainty",
        "value": baseline_uncertainty
    },

    {
        "category": "AIS rows",
        "value": ais_input_rows
    },

    {
        "category": "AIS unique MMSI",
        "value": ais_unique_mmsi
    },

    {
        "category": "AIS missing values",
        "value": ais_missing_values
    },

    {
        "category": "Compatibility",
        "value": compatibility_status
    },

    {
        "category": "Candidate ranking",
        "value": ranking_state
    },

    {
        "category": "Repeatability",
        "value": repeatability_status
    },
]


final_quality_df = pd.DataFrame(
    quality_rows
)

display(final_quality_df)

print("\nDAY 9 QUALITY SUMMARY: PASS")

Repository root: d:\SpillTrace-SIH26
Manifest loaded: True
Sensitivity report loaded: True
AIS sample loaded: True
Sensitivity report type: list
Sensitivity runs: 5


,category,value
0,Detector,UNAVAILABLE — detector status is not recorded ...
1,Geometry,"None — None features, None km², None m perimeter"
2,Geometry validity,"valid=None, empty=None"
3,Drift mode,None
4,Baseline uncertainty,None
5,AIS rows,10000
6,AIS unique MMSI,6890
7,AIS missing values,22555
8,Compatibility,None
9,Candidate ranking,None



DAY 9 QUALITY SUMMARY: PASS


In [7]:
# ============================================================
# CELL 13 — Save final frozen PPT/UI CSV
# ============================================================

from pathlib import Path
import pandas as pd

# Use the existing repository root
REPO_ROOT = Path.cwd().parents[1]

# Create final evaluation tables directory
TABLE_DIR = REPO_ROOT / "ml" / "day9_outputs" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Final frozen PPT/UI summary
ppt_csv = TABLE_DIR / "golden_scenario_quality_summary.csv"

ppt_df.to_csv(
    ppt_csv,
    index=False
)

print("PPT/UI summary saved:")
print(ppt_csv.relative_to(REPO_ROOT))

# Verify file exists
assert ppt_csv.exists(), "PPT/UI summary CSV was not created."

print("PPT/UI CSV FREEZE CHECK: PASS")

PPT/UI summary saved:
ml\day9_outputs\tables\golden_scenario_quality_summary.csv
PPT/UI CSV FREEZE CHECK: PASS


In [10]:
# ============================================================
# CELL 13 — FINAL FREEZE PROVENANCE
# No dependency on previous path variables
# ============================================================

from pathlib import Path
import hashlib
import json

# ------------------------------------------------------------
# 1. Repository root
# ------------------------------------------------------------

REPO_ROOT = Path.cwd().parents[1]

print("Repository root:", REPO_ROOT)


# ------------------------------------------------------------
# 2. Define ALL required paths directly
# ------------------------------------------------------------

SCENARIO_MANIFEST = (
    REPO_ROOT / "data" / "manifests" / "scenario_manifest.json"
)

DRIFT_METADATA = (
    REPO_ROOT / "ml" / "day5_outputs" / "drift_metadata.json"
)

SENSITIVITY_REPORT = (
    REPO_ROOT / "ml" / "day5_outputs" / "sensitivity_report.json"
)

AIS_SAMPLE = (
    REPO_ROOT / "data" / "ais" / "ais_sample_10000.csv"
)

FREEZE_DIR = (
    REPO_ROOT / "ml" / "day9_outputs" / "freeze"
)

FREEZE_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 3. Verify source files
# ------------------------------------------------------------

SOURCE_FILES = {
    "scenario_manifest": SCENARIO_MANIFEST,
    "drift_metadata": DRIFT_METADATA,
    "sensitivity_report": SENSITIVITY_REPORT,
    "ais_sample": AIS_SAMPLE,
}

print("\nSOURCE FILE CHECK")

for name, path in SOURCE_FILES.items():
    print(f"{name}: {path.exists()}")

    if not path.exists():
        print("MISSING:", path)


# ------------------------------------------------------------
# 4. Load source JSON files directly
# ------------------------------------------------------------

with open(SCENARIO_MANIFEST, "r", encoding="utf-8") as f:
    manifest = json.load(f)

with open(DRIFT_METADATA, "r", encoding="utf-8") as f:
    drift_metadata = json.load(f)

with open(SENSITIVITY_REPORT, "r", encoding="utf-8") as f:
    sensitivity_data = json.load(f)


# ------------------------------------------------------------
# 5. Golden scenario
# ------------------------------------------------------------

GOLDEN_SCENARIO = manifest.get(
    "spill_id",
    "SPILL_TEST3_001"
)


# ------------------------------------------------------------
# 6. Unsupported metrics
# ------------------------------------------------------------

unsupported_metrics = {
    "Recall@1": "NOT COMPUTED",
    "Recall@3": "NOT COMPUTED",
}


# ------------------------------------------------------------
# 7. SHA256 helper
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# 8. Freeze existing Day 9 assets
# ------------------------------------------------------------

frozen_assets = []

for path in FREEZE_DIR.rglob("*"):

    if path.is_file():

        frozen_assets.append({
            "path": str(path.relative_to(REPO_ROOT)),
            "sha256": sha256_file(path),
            "size_bytes": path.stat().st_size,
        })


# ------------------------------------------------------------
# 9. Provenance record
# ------------------------------------------------------------

provenance = {

    "freeze_day": 9,

    "golden_scenario": GOLDEN_SCENARIO,

    "manifest_version": manifest.get(
        "manifest_version"
    ),

    "scoring_version": drift_metadata.get(
        "scoring_version"
    ),

    "source_files": {
        name: str(path.relative_to(REPO_ROOT))
        for name, path in SOURCE_FILES.items()
    },

    "frozen_assets": frozen_assets,

    "unsupported_metrics": unsupported_metrics,

    "generated_from_project_outputs": True,
}


# ------------------------------------------------------------
# 10. Save provenance manifest
# ------------------------------------------------------------

provenance_path = (
    FREEZE_DIR / "freeze_provenance.json"
)

with open(
    provenance_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        provenance,
        f,
        indent=2,
        default=str
    )


# ------------------------------------------------------------
# 11. Final verification
# ------------------------------------------------------------

assert provenance_path.exists()

print("\n" + "=" * 60)
print("DAY 9 PROVENANCE FREEZE CHECK: PASS")
print("=" * 60)

print(
    "Golden scenario:",
    GOLDEN_SCENARIO
)

print(
    "Manifest version:",
    manifest.get("manifest_version")
)

print(
    "Scoring version:",
    drift_metadata.get("scoring_version")
)

print(
    "Frozen directory:",
    FREEZE_DIR.relative_to(REPO_ROOT)
)

print(
    "Provenance manifest:",
    provenance_path.relative_to(REPO_ROOT)
)

print(
    "Frozen assets:",
    len(frozen_assets)
)

print("=" * 60)

Repository root: d:\SpillTrace-SIH26

SOURCE FILE CHECK
scenario_manifest: True
drift_metadata: True
sensitivity_report: True
ais_sample: True

DAY 9 PROVENANCE FREEZE CHECK: PASS
Golden scenario: SPILL_TEST3_001
Manifest version: 1.0
Scoring version: None
Frozen directory: ml\day9_outputs\freeze
Provenance manifest: ml\day9_outputs\freeze\freeze_provenance.json
Frozen assets: 0


In [21]:
# ============================================================
# DAY 9 — LOCATE EXISTING REPEATABILITY / UNSUPPORTED EVIDENCE
# ============================================================

from pathlib import Path

print("Repository:", REPO_ROOT)
print("\nSearching for repeatability-related files...\n")

repeatability_candidates = []

for p in REPO_ROOT.rglob("*"):
    if p.is_file():
        name = p.name.lower()

        if (
            "repeat" in name
            or "reproduc" in name
            or "unsupported" in name
            or "recall" in name
        ):
            repeatability_candidates.append(p)

if repeatability_candidates:

    for p in repeatability_candidates:
        print("✓", p.relative_to(REPO_ROOT))

else:
    print("NO REPEATABILITY / UNSUPPORTED / RECALL FILES FOUND.")


print("\n" + "=" * 60)
print("DAY OUTPUT DIRECTORIES")
print("=" * 60)

for d in REPO_ROOT.rglob("*day*_outputs"):
    if d.is_dir():
        print("\nDIR:", d.relative_to(REPO_ROOT))

        files = [
            p for p in d.rglob("*")
            if p.is_file()
        ]

        for p in files:
            print("   ", p.relative_to(REPO_ROOT))

Repository: d:\SpillTrace-SIH26

Searching for repeatability-related files...

✓ .venv\Lib\site-packages\pandas\tests\indexes\datetimes\methods\test_repeat.py
✓ .venv\Lib\site-packages\pandas\tests\indexes\datetimes\methods\__pycache__\test_repeat.cpython-312.pyc
✓ .venv\Lib\site-packages\pandas\tests\indexes\period\methods\test_repeat.py
✓ .venv\Lib\site-packages\pandas\tests\indexes\period\methods\__pycache__\test_repeat.cpython-312.pyc
✓ .venv\Lib\site-packages\pandas\tests\indexes\timedeltas\methods\test_repeat.py
✓ .venv\Lib\site-packages\pandas\tests\indexes\timedeltas\methods\__pycache__\test_repeat.cpython-312.pyc
✓ .venv\Lib\site-packages\pandas\tests\io\parser\test_unsupported.py
✓ .venv\Lib\site-packages\pandas\tests\io\parser\__pycache__\test_unsupported.cpython-312.pyc
✓ .venv\Lib\site-packages\pandas\tests\series\methods\test_repeat.py
✓ .venv\Lib\site-packages\pandas\tests\series\methods\__pycache__\test_repeat.cpython-312.pyc
✓ analytics\evaluation\day9_frozen\tables\re

In [23]:
# ============================================================
# DAY 9 — LOAD ACTUAL FROZEN EVIDENCE
# ============================================================

from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd().parents[1]

FROZEN_TABLE_DIR = (
    REPO_ROOT
    / "analytics"
    / "evaluation"
    / "day9_frozen"
    / "tables"
)

REPEATABILITY_CSV = (
    FROZEN_TABLE_DIR
    / "repeatability_check.csv"
)

UNSUPPORTED_CSV = (
    FROZEN_TABLE_DIR
    / "unsupported_metrics_status.csv"
)

# ------------------------------------------------------------
# Verify actual files
# ------------------------------------------------------------

assert REPEATABILITY_CSV.exists(), (
    f"Missing: {REPEATABILITY_CSV}"
)

assert UNSUPPORTED_CSV.exists(), (
    f"Missing: {UNSUPPORTED_CSV}"
)

# ------------------------------------------------------------
# Load actual frozen evidence
# ------------------------------------------------------------

repeatability_df = pd.read_csv(REPEATABILITY_CSV)
unsupported_df = pd.read_csv(UNSUPPORTED_CSV)

print("REPEATABILITY EVIDENCE")
print("=" * 60)
display(repeatability_df)

print("\nUNSUPPORTED METRICS EVIDENCE")
print("=" * 60)
display(unsupported_df)

print("\nRepeatability columns:")
print(list(repeatability_df.columns))

print("\nUnsupported-metrics columns:")
print(list(unsupported_df.columns))

REPEATABILITY EVIDENCE


,baseline_scenario,repeatability_scenario,same_predicted_origin_lat,same_predicted_origin_lon,same_uncertainty_radius_m,repeatability_status
0,Baseline Run,Reproducibility Check (Same Seed),True,True,True,PASS



UNSUPPORTED METRICS EVIDENCE


,metric,status
0,Recall@1,NOT COMPUTED
1,Recall@3,NOT COMPUTED
2,ground_truth_verified_attribution,False



Repeatability columns:
['baseline_scenario', 'repeatability_scenario', 'same_predicted_origin_lat', 'same_predicted_origin_lon', 'same_uncertainty_radius_m', 'repeatability_status']

Unsupported-metrics columns:
['metric', 'status']


In [28]:
# ============================================================
# DAY 9 — FINALIZE CORRECT FROZEN EVALUATION ASSETS
# ============================================================

from pathlib import Path
import json
import hashlib
import shutil
import pandas as pd

# ------------------------------------------------------------
# 1. Repository
# ------------------------------------------------------------

REPO_ROOT = Path.cwd().parents[1]

FROZEN_DIR = (
    REPO_ROOT
    / "analytics"
    / "evaluation"
    / "day9_frozen"
)

FROZEN_TABLE_DIR = FROZEN_DIR / "tables"

FROZEN_DIR.mkdir(parents=True, exist_ok=True)
FROZEN_TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_ROOT)
print("Frozen directory:", FROZEN_DIR)


# ------------------------------------------------------------
# 2. Existing quality CSV generated earlier
# ------------------------------------------------------------

source_quality_csv = (
    REPO_ROOT
    / "ml"
    / "day9_outputs"
    / "tables"
    / "golden_scenario_quality_summary.csv"
)

assert source_quality_csv.exists(), (
    f"Existing quality CSV not found: {source_quality_csv}"
)

# Correct final frozen location
ppt_csv = (
    FROZEN_TABLE_DIR
    / "golden_scenario_quality_summary.csv"
)

shutil.copy2(source_quality_csv, ppt_csv)

print(
    "✓ Quality CSV frozen:",
    ppt_csv.relative_to(REPO_ROOT)
)


# ------------------------------------------------------------
# 3. Read quality CSV
# ------------------------------------------------------------

quality_df = pd.read_csv(ppt_csv)

display(quality_df)


# ------------------------------------------------------------
# 4. Create final quality-summary JSON
# ------------------------------------------------------------

final_summary = {
    "spill_id": "SPILL_TEST3_001",
    "source_csv": str(
        ppt_csv.relative_to(REPO_ROOT)
    ),
    "quality_summary": [
        {
            str(row["Metric"]):
            None if pd.isna(row["Value"])
            else str(row["Value"])
        }
        for _, row in quality_df.iterrows()
    ],
    "generated_from_project_outputs": True,
}

final_summary_path = (
    FROZEN_DIR
    / "golden_scenario_quality_summary.json"
)

with open(
    final_summary_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "✓ Quality JSON frozen:",
    final_summary_path.relative_to(REPO_ROOT)
)


# ------------------------------------------------------------
# 5. Existing frozen evidence
# ------------------------------------------------------------

repeatability_path = (
    FROZEN_TABLE_DIR
    / "repeatability_check.csv"
)

unsupported_path = (
    FROZEN_TABLE_DIR
    / "unsupported_metrics_status.csv"
)

assert repeatability_path.exists(), (
    f"Missing: {repeatability_path}"
)

assert unsupported_path.exists(), (
    f"Missing: {unsupported_path}"
)

print("✓ Repeatability evidence exists")
print("✓ Unsupported-metrics evidence exists")


# ------------------------------------------------------------
# 6. SHA256 helper
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# 7. Build actual frozen-assets list
# ------------------------------------------------------------

frozen_assets = []

for path in sorted(FROZEN_DIR.rglob("*")):

    if not path.is_file():
        continue

    # Do not hash the provenance file itself
    if path.name == "freeze_provenance.json":
        continue

    frozen_assets.append({
        "path": str(
            path.relative_to(REPO_ROOT)
        ),
        "sha256": sha256_file(path),
        "size_bytes": path.stat().st_size,
    })


# ------------------------------------------------------------
# 8. Source evidence
# ------------------------------------------------------------

source_files = {
    "scenario_manifest":
        REPO_ROOT
        / "data"
        / "manifests"
        / "scenario_manifest.json",

    "drift_metadata":
        REPO_ROOT
        / "ml"
        / "day5_outputs"
        / "drift_metadata.json",

    "sensitivity_report":
        REPO_ROOT
        / "ml"
        / "day5_outputs"
        / "sensitivity_report.json",

    "ais_sample":
        REPO_ROOT
        / "data"
        / "ais"
        / "ais_sample_10000.csv",
}


# ------------------------------------------------------------
# 9. Final provenance
# ------------------------------------------------------------

provenance = {
    "freeze_day": 9,
    "golden_scenario": "SPILL_TEST3_001",
    "manifest_version": "1.0",
    "scoring_version": None,

    "source_files": {
        name: str(
            path.relative_to(REPO_ROOT)
        )
        for name, path in source_files.items()
    },

    "frozen_assets": frozen_assets,

    "unsupported_metrics": {
        "Recall@1": "NOT COMPUTED",
        "Recall@3": "NOT COMPUTED",
        "ground_truth_verified_attribution": False,
    },

    "generated_from_project_outputs": True,
}


# ------------------------------------------------------------
# 10. Save provenance
# ------------------------------------------------------------

provenance_path = (
    FROZEN_DIR
    / "freeze_provenance.json"
)

with open(
    provenance_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        provenance,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 11. Final local verification
# ------------------------------------------------------------

required_assets = [
    final_summary_path,
    ppt_csv,
    repeatability_path,
    unsupported_path,
    provenance_path,
]

assert all(
    p.exists()
    for p in required_assets
)

assert len(frozen_assets) >= 4

print("\n" + "=" * 60)
print("DAY 9 FROZEN ASSETS PREPARED")
print("=" * 60)

for path in required_assets:
    print("✓", path.relative_to(REPO_ROOT))

print("\nFrozen asset count:", len(frozen_assets))
print("Provenance updated:", provenance_path.relative_to(REPO_ROOT))

Repository: d:\SpillTrace-SIH26
Frozen directory: d:\SpillTrace-SIH26\analytics\evaluation\day9_frozen
✓ Quality CSV frozen: analytics\evaluation\day9_frozen\tables\golden_scenario_quality_summary.csv


,Metric,Value
0,Scenario,SPILL_TEST3_001
1,Detector,UNAVAILABLE — detector status is not recorded ...
2,Geometry,UNAVAILABLE
3,Geometry Valid,NaN
4,Area (km²),NaN
5,Perimeter (m),NaN
6,Drift Mode,NaN
7,Drift Uncertainty (m),NaN
8,AIS Rows,10000
9,AIS Unique MMSI,6890


✓ Quality JSON frozen: analytics\evaluation\day9_frozen\golden_scenario_quality_summary.json
✓ Repeatability evidence exists
✓ Unsupported-metrics evidence exists

DAY 9 FROZEN ASSETS PREPARED
✓ analytics\evaluation\day9_frozen\golden_scenario_quality_summary.json
✓ analytics\evaluation\day9_frozen\tables\golden_scenario_quality_summary.csv
✓ analytics\evaluation\day9_frozen\tables\repeatability_check.csv
✓ analytics\evaluation\day9_frozen\tables\unsupported_metrics_status.csv
✓ analytics\evaluation\day9_frozen\freeze_provenance.json

Frozen asset count: 5
Provenance updated: analytics\evaluation\day9_frozen\freeze_provenance.json


In [16]:
import subprocess

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

 D analytics/notebooks/day7_leftover.md
?? analytics/evaluation/
?? analytics/notebooks/13_evaluation_freeze.ipynb
?? ml/day9_outputs/



In [17]:
#16
subprocess.run(
    ["git", "add", "analytics/evaluation/day9_frozen"],
    cwd=REPO_ROOT,
    check=True
)

print("=== STAGED FILES ===")
result = subprocess.run(
    ["git", "diff", "--cached", "--name-status"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

=== STAGED FILES ===
A	analytics/evaluation/day9_frozen/tables/golden_scenario_sensitivity.csv
A	analytics/evaluation/day9_frozen/tables/repeatability_check.csv
A	analytics/evaluation/day9_frozen/tables/unsupported_metrics_status.csv



In [18]:
#17
commit_message = "feat(day9): freeze golden scenario evaluation assets"

result = subprocess.run(
    ["git", "commit", "-m", commit_message],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Git commit failed.")

[main d79a431] feat(day9): freeze golden scenario evaluation assets
 3 files changed, 12 insertions(+)
 create mode 100644 analytics/evaluation/day9_frozen/tables/golden_scenario_sensitivity.csv
 create mode 100644 analytics/evaluation/day9_frozen/tables/repeatability_check.csv
 create mode 100644 analytics/evaluation/day9_frozen/tables/unsupported_metrics_status.csv




In [19]:
#19
result = subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True
)

print("=== FINAL GIT STATUS ===")
print(result.stdout if result.stdout else "WORKING TREE CLEAN")

print("\nDay 9 commit completed.")

=== FINAL GIT STATUS ===
 D analytics/notebooks/day7_leftover.md
?? analytics/notebooks/13_evaluation_freeze.ipynb
?? ml/day9_outputs/


Day 9 commit completed.
